# **Parsing all 26 rainfall .grd files into one numpy array**

In [1]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA: True
Tesla T4


In [2]:
import numpy as np
import os
from pathlib import Path

def parse_imd_rainfall(filepath, year):
    """Parse IMD 0.25° binary rainfall file"""
    import calendar
    days = 366 if calendar.isleap(year) else 365

    with open(filepath, 'rb') as f:
        data = np.fromfile(f, dtype=np.float32)

    total_expected = days * 135 * 129
    data = data[:total_expected]
    data = data.reshape(days, 129, 135)
    data[data < 0] = np.nan
    return data

DATA_DIR = Path("/kaggle/input/datasets/vikashkumar96314/imd-rainfall-dataset")
YEARS = list(range(2000, 2026))

all_data = []
for year in YEARS:
    candidates = [
        DATA_DIR / f"Rainfall_ind{year}_rfp25.grd",
        DATA_DIR / f"RF25_{year}.grd",
        DATA_DIR / f"rain_{year}.grd",
    ]
    filepath = next((p for p in candidates if p.exists()), None)

    if filepath is None:
        print(f" Missing: {year}")
        continue

    arr = parse_imd_rainfall(filepath, year)
    all_data.append(arr)
    print(f"{year} → shape {arr.shape}")

rainfall_full = np.concatenate(all_data, axis=0)
print(f"\nFull dataset shape: {rainfall_full.shape}")

# Saving the .npy
save_dir = "/kaggle/working/"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(
    save_dir,
    "rainfall_2000_2025.npy"
)

np.save(save_path, rainfall_full)

print("Saved to:", save_path)
print("Saved.")

2000 → shape (366, 129, 135)
2001 → shape (365, 129, 135)
2002 → shape (365, 129, 135)
2003 → shape (365, 129, 135)
2004 → shape (366, 129, 135)
2005 → shape (365, 129, 135)
2006 → shape (365, 129, 135)
2007 → shape (365, 129, 135)
2008 → shape (366, 129, 135)
2009 → shape (365, 129, 135)
2010 → shape (365, 129, 135)
2011 → shape (365, 129, 135)
2012 → shape (366, 129, 135)
2013 → shape (365, 129, 135)
2014 → shape (365, 129, 135)
2015 → shape (365, 129, 135)
2016 → shape (366, 129, 135)
2017 → shape (365, 129, 135)
2018 → shape (365, 129, 135)
2019 → shape (365, 129, 135)
2020 → shape (366, 129, 135)
2021 → shape (365, 129, 135)
2022 → shape (365, 129, 135)
2023 → shape (365, 129, 135)
2024 → shape (366, 129, 135)
2025 → shape (365, 129, 135)

Full dataset shape: (9497, 129, 135)
Saved to: /kaggle/working/rainfall_2000_2025.npy
Saved.


# **Building sliding window sequences for training**

In [3]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# Load rainfall dataset
rainfall = np.load(
    "/kaggle/working/rainfall_2000_2025.npy"
)

# Replace NaN and normalize
rainfall = np.nan_to_num(rainfall, nan=0.0)

# normalize rainfall (IMD max around 500mm/day)
rainfall = (rainfall / 500.0).astype(np.float32)

print("Rainfall loaded:", rainfall.shape)


LOOKBACK = 30   # past 30 days
HORIZON  = 7    # predict next 7 days

total_days = len(rainfall)


# Correct split considering leap years
train_end = 7305        # 2000-2019
val_end   = 8036        # 2020-2021


class RainfallDataset(Dataset):

    def __init__(self, data, start, end, lookback, horizon):

        self.data = data
        self.lookback = lookback
        self.horizon = horizon

        self.indices = list(
            range(
                start,
                end - lookback - horizon + 1
            )
        )


    def __len__(self):
        return len(self.indices)


    def __getitem__(self, idx):

        i = self.indices[idx]

        # input:
        # 30 days rainfall maps
        X = self.data[
            i : i + self.lookback
        ]

        # output:
        # next 7 days rainfall maps
        y = self.data[
            i + self.lookback :
            i + self.lookback + self.horizon
        ]


        # ConvLSTM format:
        # (time, channel, height, width)

        X = torch.FloatTensor(X).unsqueeze(1)

        y = torch.FloatTensor(y).unsqueeze(1)


        return X, y

train_ds = RainfallDataset(
    rainfall,
    0,
    train_end,
    LOOKBACK,
    HORIZON
)

val_ds = RainfallDataset(
    rainfall,
    train_end,
    val_end,
    LOOKBACK,
    HORIZON
)

test_ds = RainfallDataset(
    rainfall,
    val_end,
    total_days,
    LOOKBACK,
    HORIZON
)


print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))
print("Test samples:", len(test_ds))


train_loader = DataLoader(
    train_ds,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# Check shape
X, y = next(iter(train_loader))

print("Input shape :", X.shape)
print("Output shape:", y.shape)

Rainfall loaded: (9497, 129, 135)
Train samples: 7269
Val samples: 695
Test samples: 1425
Input shape : torch.Size([4, 30, 1, 129, 135])
Output shape: torch.Size([4, 7, 1, 129, 135])


# **The ConvLSTM2D model**

In [4]:
import torch
import torch.nn as nn


class ConvLSTMCell(nn.Module):

    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super().__init__()

        padding = kernel_size // 2

        self.hidden_channels = hidden_channels

        self.conv = nn.Conv2d(
            in_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size,
            padding=padding
        )


    def forward(self, x, h, c):

        combined = torch.cat([x, h], dim=1)

        gates = self.conv(combined)

        i, f, g, o = gates.chunk(4, dim=1)

        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)

        c = f * c + i * g
        h = o * torch.tanh(c)

        return h, c



class RainfallConvLSTM(nn.Module):

    def __init__(self, hidden_channels=32, horizon=7):
        super().__init__()

        self.horizon = horizon
        self.hidden_channels = hidden_channels


        self.encoder1 = ConvLSTMCell(
            1,
            hidden_channels
        )

        self.encoder2 = ConvLSTMCell(
            hidden_channels,
            hidden_channels
        )


        self.output_conv = nn.Sequential(

            nn.Conv2d(
                hidden_channels,
                32,
                3,
                padding=1
            ),

            nn.ReLU(),

            nn.Conv2d(
                32,
                horizon,
                1
            ),

            nn.ReLU()
        )


    def forward(self, x):

        # x:
        # B,T,C,H,W

        B,T,C,H,W = x.shape


        h1 = torch.zeros(
            B,
            self.hidden_channels,
            H,
            W,
            device=x.device
        )

        c1 = torch.zeros_like(h1)


        h2 = torch.zeros(
            B,
            self.hidden_channels,
            H,
            W,
            device=x.device
        )

        c2 = torch.zeros_like(h2)



        for t in range(T):

            xt = x[:,t]


            h1,c1 = self.encoder1(
                xt,
                h1,
                c1
            )


            h2,c2 = self.encoder2(
                h1,
                h2,
                c2
            )


        out = self.output_conv(h2)

        # B,7,H,W
        return out



# ===================
# TEST
# ===================

model = RainfallConvLSTM(
    hidden_channels=32,
    horizon=7
)


dummy = torch.randn(
    2,
    30,
    1,
    129,
    135
)


pred = model(dummy)

print(pred.shape)

torch.Size([2, 7, 129, 135])


**Optimizer**

# **Training loop**

In [5]:
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
import gc


# =========================
# CLEAN GPU MEMORY
# =========================

gc.collect()
torch.cuda.empty_cache()


# =========================
# DEVICE
# =========================

device = torch.device("cuda")

print("Training on:", device)
print(torch.cuda.get_device_name(0))
print(torch.cuda.memory_summary())


# =========================
# MODEL
# =========================

model = RainfallConvLSTM(
    hidden_channels=32,
    horizon=7
).to(device)


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

criterion = nn.MSELoss()


# AMP
scaler = GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)


SAVE_PATH = "/kaggle/working/rainfall_convlstm_best.pth"

best_val_loss = float("inf")


# =========================
# TRAINING
# =========================

EPOCHS = 30


for epoch in range(EPOCHS):

    # ---------------------
    # TRAIN
    # ---------------------

    model.train()

    train_loss = 0


    for batch_idx, (X_batch, y_batch) in enumerate(train_loader):

        X_batch = X_batch.to(
            device,
            non_blocking=True
        )

        y_batch = y_batch.to(
            device,
            non_blocking=True
        )


        # (B,7,1,H,W)
        # ->
        # (B,7,H,W)

        y_batch = y_batch.squeeze(2)


        optimizer.zero_grad()


        # AMP CUDA
        with autocast(
            "cuda",
            enabled=True
        ):

            pred = model(X_batch)

            loss = criterion(
                pred,
                y_batch
            )


        scaler.scale(loss).backward()


        # gradient clipping

        scaler.unscale_(optimizer)


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )


        scaler.step(optimizer)

        scaler.update()


        train_loss += loss.item()



        if batch_idx % 10 == 0:

            print(
                f"Epoch {epoch+1}/{EPOCHS} "
                f"| Batch {batch_idx}/{len(train_loader)} "
                f"| Loss {loss.item():.6f}"
            )



    # ---------------------
    # VALIDATION
    # ---------------------

    model.eval()

    val_loss = 0


    with torch.no_grad():

        for X_batch, y_batch in val_loader:


            X_batch = X_batch.to(
                device,
                non_blocking=True
            )


            y_batch = y_batch.to(
                device,
                non_blocking=True
            )


            y_batch = y_batch.squeeze(2)



            with autocast(
                "cuda",
                enabled=True
            ):


                pred = model(X_batch)


                loss = criterion(
                    pred,
                    y_batch
                )


            val_loss += loss.item()



    train_loss /= len(train_loader)

    val_loss /= len(val_loader)



    train_rmse = (train_loss ** 0.5) * 500

    val_rmse = (val_loss ** 0.5) * 500



    print("\n==============================")

    print(f"Epoch {epoch+1:02d} DONE")

    print(
        f"Train RMSE : {train_rmse:.2f} mm/day"
    )

    print(
        f"Val RMSE   : {val_rmse:.2f} mm/day"
    )


    # ---------------------
    # SAVE BEST
    # ---------------------

    if val_loss < best_val_loss:


        best_val_loss = val_loss


        torch.save(
            model.state_dict(),
            SAVE_PATH
        )

        print("Best model saved")


print("\n==============================")

print("Training complete")

print("Saved model:")

print(SAVE_PATH)

Training on: cuda
Tesla T4
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   